# Preprocessors

Preprocessors are optional. If none is configured, observations pass through unchanged. Use `@preprocess` when records need Python logic before schema queries run.

Use a query when the source shape is stable and selection is enough. Use a preprocessor when you need type coercion, renaming, source-specific cleanup, windowing, session splitting, or derived fields. The preprocessor prepares records; the schema remains the model-facing contract.

This guide uses Iris data, but starts from a deliberately awkward raw shape: feature names live in a nested dictionary and the label lives outside that feature object.


In [1]:
import polars as pl
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()

raw_records = pl.read_ndjson("docs/data/iris-raw.jsonl").head(3).to_dicts()


In [2]:
pprint(raw_records[0])


{
│   'features': {
│   │   'sepal length (cm)': 5.1,
│   │   'sepal width (cm)': 3.5,
│   │   'petal length (cm)': 1.4,
│   │   'petal width (cm)': 0.2
│   },
│   'species': 'setosa'
}

A transformation preprocessor returns one normalized record for each input record. It is the right choice for type coercion, renaming, flattening, or attaching derived fields. Pass it to `PolarsDataModule.from_model(..., preprocessor=...)` or configure it on a dataset so training, prediction, and serving share the same transformation.

In [3]:
@j2v.preprocess
def simplify_iris(record: dict) -> dict:
    features = record["features"]
    return {
        "sepal_length": float(features["sepal length (cm)"]),
        "petal_length": float(features["petal length (cm)"]),
        "species": record["species"],
    }


In [4]:
pprint(simplify_iris(raw_records[0]))


{'sepal_length': 5.1, 'petal_length': 1.4, 'species': 'setosa'}

A yielding preprocessor can expand one input into many outputs. That pattern is useful when one account history should become multiple windows, one session log should become multiple training observations, or one raw export contains many model records.

In [5]:
@j2v.preprocess(yields=True)
def iris_records(records):
    for record in records:
        yield simplify_iris(record)


In [6]:
pprint(list(iris_records(raw_records)))


[
│   {'sepal_length': 5.1, 'petal_length': 1.4, 'species': 'setosa'},
│   {'sepal_length': 7.0, 'petal_length': 4.7, 'species': 'versicolor'},
│   {'sepal_length': 6.3, 'petal_length': 6.0, 'species': 'virginica'}
]